<a href="https://colab.research.google.com/github/Najma-1437/PS-Agents/blob/main/UCS_Trailer.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Group: 4
## Members:
### 191231 - Patrick Mungai
### 192082 - DavidTelios Muriithi
### 192797 - Ahmed Najma Wanjiku

The objective is to implement **Uniform Cost Search (UCS)** to determine the minimum-distance route for a self-driving trailer travelling from **Nairobi to Kisumu**.

Using the simplified, weighted, directed highway network developed in **Practical Exercise 3 Part 1**.

Each town is represented as a **node (state)**, each road connection as a **directed edge (action)**, and each edge weight as the **distance in kilometres** between two connected towns.

### 1.2 Search Problem Components

| Component | Definition |
|---|---|
| **Agent** | Self-driving trailer |
| **Initial State** | Nairobi |
| **Goal State** | Kisumu |
| **States** | Towns in the simplified highway network |
| **Actions** | Travel along an available directed road connection |
| **Transition Model** | Move from the current town to a directly connected town |
| **Path Cost** | Sum of road distances in kilometres |
| **Goal Test** | Check whether the current town is Kisumu |



Uniform Cost Search expands the frontier node with the lowest accumulated path cost, denoted by:

$$
g(n) = \sum_{i=1}^{k} c(e_i)
$$

Where:

$g(n)$ is the cumulative cost of reaching node $n$.

$c(e_i)$ is the distance associated with edge $e_i$.

 $k$ is the number of edges traversed along the path.



In [1]:
graph = {
    "Nairobi": { #current town, then towns reachable from Nairobi
        "Naivasha": 90,
        "Narok": 142,
        "Thika": 40
    }
}

In [2]:
print(graph["Nairobi"])

{'Naivasha': 90, 'Narok': 142, 'Thika': 40}


In [3]:
print(graph["Nairobi"]["Naivasha"]) #Accessing the cost of one connection

90


In [4]:
#Completed graph dictionary

graph = {
    "Nairobi": { #initial state
        "Naivasha": 90,
        "Narok": 142,
        "Thika": 40
    },

    "Naivasha": {
        "Nakuru": 71
    },

    "Narok": {
        "Bomet": 81
    },

    "Thika": {
        "Nyeri": 110
    },

    "Nyeri": {
        "Nyahururu": 103
    },

    "Nyahururu": {
        "Nakuru": 75
    },

    "Nakuru": {
        "Kericho": 108
    },

    "Bomet": {
        "Kericho": 79
    },

    "Kericho": {
        "Kisumu": 84
    },

    "Kisumu": {} #our destination, so no outgoing connections
}


In [5]:
start = "Nairobi"
goal = "Kisumu"

print("Initial state:", start)
print("Goal state:", goal)

Initial state: Nairobi
Goal state: Kisumu


In [6]:
#verifying our graph
number_of_towns = len(graph)

number_of_edges = sum(
    len(neighbors)
    for neighbors in graph.values()
)

print("Number of towns:", number_of_towns)
print("Number of directed edges:", number_of_edges)

Number of towns: 10
Number of directed edges: 11


In [7]:
import heapq
import itertools

In [8]:
   def uniform_cost_search(graph, start, goal, verbose=True):
       """
       Uniform Cost Search on a weighted, directed graph.

       Returns: (path, total_cost, expanded_nodes)
                path is None and total_cost is float('inf') if no route exists.
       """
       counter = itertools.count()              # tie-breaker so heapq never compares nodes
       frontier = [(0, next(counter), start)]   # priority queue of (g(n), tie, node)
       best_cost = {start: 0}                   # cheapest g(n) found so far for each node
       parent = {start: None}                   # for path reconstruction
       explored = set()                         # nodes already expanded
       expanded_order = []                      # nodes expanded, in order

       step = 0
       while frontier:
           g, _, node = heapq.heappop(frontier)  # lowest cumulative cost

           # Skip stale entries (a cheaper route to this node was already expanded)
           if node in explored:
               continue

           step += 1
           explored.add(node)
           expanded_order.append(node)

           if verbose:
               print(f"Step {step}: expand {node} (g = {g} km)")

           # Goal test when a node is POPPED, which guarantees optimality
           if node == goal:
               path = []
               current = node
               while current is not None:
                   path.append(current)
                   current = parent[current]
               path.reverse()
               if verbose:
                   print(f"\nGoal '{goal}' reached.")
               return path, g, expanded_order

           # Generate successors
           for neighbor, step_cost in graph[node].items():
               new_cost = g + step_cost
               if neighbor not in explored and new_cost < best_cost.get(neighbor, float("inf")):
                   best_cost[neighbor] = new_cost
                   parent[neighbor] = node
                   heapq.heappush(frontier, (new_cost, next(counter), neighbor))

           if verbose:
               # Live frontier, skipping stale entries, sorted by cost
               live = sorted(
                   (c, n) for c, _, n in frontier
                   if n not in explored and c == best_cost[n]
               )
               print("   Frontier:", [(n, c) for c, n in live])

       return None, float("inf"), expanded_order

## Execution

In [9]:
path, total_cost, expanded = uniform_cost_search(graph, start, goal)

Step 1: expand Nairobi (g = 0 km)
   Frontier: [('Thika', 40), ('Naivasha', 90), ('Narok', 142)]
Step 2: expand Thika (g = 40 km)
   Frontier: [('Naivasha', 90), ('Narok', 142), ('Nyeri', 150)]
Step 3: expand Naivasha (g = 90 km)
   Frontier: [('Narok', 142), ('Nyeri', 150), ('Nakuru', 161)]
Step 4: expand Narok (g = 142 km)
   Frontier: [('Nyeri', 150), ('Nakuru', 161), ('Bomet', 223)]
Step 5: expand Nyeri (g = 150 km)
   Frontier: [('Nakuru', 161), ('Bomet', 223), ('Nyahururu', 253)]
Step 6: expand Nakuru (g = 161 km)
   Frontier: [('Bomet', 223), ('Nyahururu', 253), ('Kericho', 269)]
Step 7: expand Bomet (g = 223 km)
   Frontier: [('Nyahururu', 253), ('Kericho', 269)]
Step 8: expand Nyahururu (g = 253 km)
   Frontier: [('Kericho', 269)]
Step 9: expand Kericho (g = 269 km)
   Frontier: [('Kisumu', 353)]
Step 10: expand Kisumu (g = 353 km)

Goal 'Kisumu' reached.


In [10]:
print("Expansion order:", " -> ".join(expanded))
print("Nodes expanded :", len(expanded))

Expansion order: Nairobi -> Thika -> Naivasha -> Narok -> Nyeri -> Nakuru -> Bomet -> Nyahururu -> Kericho -> Kisumu
Nodes expanded : 10


## Optimal Route

In [11]:
print("Optimal path:", " -> ".join(path))
running = 0
for a, b in zip(path, path[1:]):
    running += graph[a][b]
    print(f"{a:<10} -> {b:<10} {graph[a][b]:>4} km (cumulative {running} km)")
print("Total distance:", total_cost, "km")

Optimal path: Nairobi -> Naivasha -> Nakuru -> Kericho -> Kisumu
Nairobi    -> Naivasha     90 km (cumulative 90 km)
Naivasha   -> Nakuru       71 km (cumulative 161 km)
Nakuru     -> Kericho     108 km (cumulative 269 km)
Kericho    -> Kisumu       84 km (cumulative 353 km)
Total distance: 353 km


## Validation

In [12]:
def route_cost(route):
    return sum(graph[a][b] for a, b in zip(route, route[1:]))

part1_routes = [
    ["Nairobi","Naivasha","Nakuru","Kericho","Kisumu"],
    ["Nairobi","Narok","Bomet","Kericho","Kisumu"],
    ["Nairobi","Thika","Nyeri","Nyahururu","Nakuru","Kericho","Kisumu"],
]
for r in part1_routes:
    print(" -> ".join(r), route_cost(r), "km")

assert path == min(part1_routes, key=route_cost)
assert total_cost == min(route_cost(r) for r in part1_routes)
print("Validation passed: UCS matches Part 1 (353 km).")

Nairobi -> Naivasha -> Nakuru -> Kericho -> Kisumu 353 km
Nairobi -> Narok -> Bomet -> Kericho -> Kisumu 386 km
Nairobi -> Thika -> Nyeri -> Nyahururu -> Nakuru -> Kericho -> Kisumu 520 km
Validation passed: UCS matches Part 1 (353 km).


## Test Edge Cases

In [13]:
def run_test(label, s, g):
    print(f"\n--- {label}: {s} -> {g} ---")
    if s not in graph or g not in graph:
        print("INVALID INPUT (town not in network)"); return None, None
    p, c, _ = uniform_cost_search(graph, s, g, verbose=False)
    print("UNREACHABLE" if p is None else f"{' -> '.join(p)} | {c} km")
    return p, c

run_test("Other destination", "Nairobi", "Nakuru")    # expect 161 km via Naivasha
run_test("Other destination", "Nairobi", "Kericho")   # expect 269 km
run_test("Start = goal", "Nairobi", "Nairobi")        # expect cost 0
run_test("Unreachable (directed)", "Kisumu", "Nairobi")
run_test("Unreachable", "Narok", "Thika")
run_test("Unknown town", "Nairobi", "Eldoret")


--- Other destination: Nairobi -> Nakuru ---
Nairobi -> Naivasha -> Nakuru | 161 km

--- Other destination: Nairobi -> Kericho ---
Nairobi -> Naivasha -> Nakuru -> Kericho | 269 km

--- Start = goal: Nairobi -> Nairobi ---
Nairobi | 0 km

--- Unreachable (directed): Kisumu -> Nairobi ---
UNREACHABLE

--- Unreachable: Narok -> Thika ---
UNREACHABLE

--- Unknown town: Nairobi -> Eldoret ---
INVALID INPUT (town not in network)


(None, None)

## Findings

**Result.** UCS found Nairobi → Naivasha → Nakuru → Kericho → Kisumu at a total cost of **353 km**. This matches the Part 1 calculation and a brute-force enumeration of all three possible routes (353, 386 and 520 km).

**How traversal works.** UCS expands the frontier node with the lowest cumulative cost g(n): Nairobi (0), Thika (40), Naivasha (90), Narok (142), Nyeri (150), Nakuru (161), Bomet (223), Nyahururu (253), Kericho (269), Kisumu (353). It orders by distance travelled which is why Thika (40 km) was expanded before Naivasha (90 km) even though it leads to the worst route.

**Key decision points.**
- Nakuru was reached at 161 km via Naivasha. The Thika → Nyeri → Nyahururu route would reach it at 328 km, so it was never used. When Nyahururu was expanded, Nakuru was already explored and skipped.
- Kericho was first reached via Nakuru at 269 km. When Bomet was expanded, the alternative Bomet → Kericho gave 223 + 79 = 302 km, which is worse, so it was discarded.

**Why UCS is optimal.** The goal test runs when a node is *popped*, not when it is generated. Since all step costs are positive and nodes are popped in non-decreasing order of g(n), the first time Kisumu is popped no cheaper path can exist: any undiscovered route must pass through a frontier node whose cost is already at least 353 km. UCS is also complete, because it finds a solution whenever one exists.

**Edge cases.** Unreachable goals (e.g. Kisumu → Nairobi, because roads are directed) exhaust the frontier and return no path. Start = goal returns cost 0. Towns outside the network are rejected.

**Limitations.** UCS has no sense of where the goal lies, so it expanded all 10 towns, including Thika and Narok, which lead away from the best route. Time and memory grow with the number of nodes cheaper than the optimal cost. A heuristic search such as A* would reduce this effort.